# 설비 센서 데이터 불러오기 예제

`data_loader.load_sensor_data`로 MinIO(또는 로컬)의 설비 센서 원본을 받아
**설비 1대 × 1초 단위 표**로 변환하고, 간단히 살펴본 뒤 저장하는 예제입니다.

**실행 전 준비**
1. 저장소 최상위 폴더에 `.env` 작성 (`.env.example` 참고, README 2장)
2. 패키지 설치: `pip install -r requirements.txt`
3. 노트북 실행용 패키지(별도 설치): `ipykernel`, `matplotlib`
4. 아래 **2. 조회 조건** 셀의 `<...>` 값을 실제 값으로 변경

## 1. 준비

노트북이 `examples/` 폴더에 있으므로, 상위 폴더의 `data_loader.py`를 import할 수 있게 경로를 추가합니다.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# data_loader.py가 있는 저장소 최상위 폴더를 찾아 import 경로에 추가
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_loader.py").exists())
sys.path.insert(0, str(REPO_ROOT))

from data_loader import convert_sensor_df, load_sensor_data  # noqa: E402

print("저장소 경로:", REPO_ROOT)

In [ ]:
# .env가 제대로 읽혔는지 확인 (값은 출력하지 않고 설정 여부만 표시)
for name in ["MINIO_ENDPOINT", "MINIO_BUCKET", "MINIO_ACCESS_KEY", "MINIO_SECRET_KEY", "MINIO_SECURE"]:
    print(f"{name:18s}: {'설정됨' if os.getenv(name) else '없음'}")

## 2. 조회 조건

| 변수 | 설명 | 예시 |
|---|---|---|
| `DATA_PATH` | MinIO 경로(`minio://<버킷>/<prefix>/`) 또는 로컬 폴더 | `minio://<버킷>/<사이트>/FDC/TRC/202611/` |
| `EQP_ID` | 설비 ID (파일명에 포함된 문자열) | `<설비ID>` |
| `START_TIME`, `END_TIME` | 조회 기간 (양 끝 포함) | `2026-11-15 08:00:00` |
| `COLUMNS` | 가져올 센서 목록, `None`이면 전체 | `["LINE_SPEED", "EQP_STATE"]` |

> 메모리 주의: 하루치 파일 전체를 읽은 뒤 거르므로, 짧은 기간·필요한 센서만으로 먼저 시험하세요.

In [ ]:
DATA_PATH = "minio://<버킷>/<경로>/"   # TODO: 실제 경로로 변경
EQP_ID = "<설비ID>"                    # TODO: 실제 설비 ID로 변경
START_TIME = "2026-11-15 08:00:00"
END_TIME = "2026-11-15 12:00:00"
COLUMNS = ["LINE_SPEED", "EQP_STATE"]  # None이면 전체 센서

if "<" in DATA_PATH or "<" in EQP_ID:
    raise ValueError("DATA_PATH와 EQP_ID의 <...> 부분을 실제 값으로 바꾼 뒤 실행하세요.")

## 3. 데이터 불러오기

In [ ]:
df = load_sensor_data(
    data_path=DATA_PATH,
    eqp_id=EQP_ID,
    start_time=START_TIME,
    end_time=END_TIME,
    columns=COLUMNS,
)
print(df.shape)
df.head()

## 4. 숫자 변환

센서값은 **문자열**로 반환됩니다. 계산·그래프에 쓰려면 숫자로 바꿉니다.
숫자로 바꿀 수 없는 값은 `NaN`이 됩니다(`errors="coerce"`).

In [ ]:
sensor_cols = [c for c in df.columns if c not in ("eqp_id", "Sensor Time")]
before_na = df[sensor_cols].isna().sum()
df[sensor_cols] = df[sensor_cols].apply(pd.to_numeric, errors="coerce")

print("숫자 변환으로 새로 생긴 결측:", (df[sensor_cols].isna().sum() - before_na).to_dict())
df.dtypes

## 5. 살펴보기

In [ ]:
print("기간:", df["Sensor Time"].min(), "~", df["Sensor Time"].max())
print("행 수:", len(df))
summary = pd.DataFrame({
    "값 있는 행": df[sensor_cols].notna().sum(),
    "결측 행": df[sensor_cols].isna().sum(),
    "결측률(%)": (df[sensor_cols].isna().mean() * 100).round(2),
})
summary

In [ ]:
df[sensor_cols].describe().T

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(sensor_cols), 1, figsize=(12, 2.5 * len(sensor_cols)), sharex=True)
axes = [axes] if len(sensor_cols) == 1 else axes
for ax, col in zip(axes, sensor_cols):
    ax.plot(df["Sensor Time"], df[col], linewidth=0.8)
    ax.set_ylabel(col)
    ax.grid(alpha=0.3)
axes[0].set_title(f"{EQP_ID}  {START_TIME} ~ {END_TIME}")
plt.tight_layout()
plt.show()

## 6. 결과 저장

`output/<기준일>/<설비ID>_sensor_<시작시각>-<종료시각>.parquet`으로 저장합니다. 같은 조건으로 다시 실행하면 **같은 파일을 덮어쓰므로** 중복이 생기지 않습니다.
(`output/`은 `.gitignore`에 등록되어 Git에 올라가지 않습니다.)

In [ ]:
base_date = pd.Timestamp(START_TIME).strftime("%Y-%m-%d")
out_dir = REPO_ROOT / "output" / base_date
out_dir.mkdir(parents=True, exist_ok=True)
start_s, end_s = (pd.Timestamp(t).strftime("%H%M%S") for t in (START_TIME, END_TIME))
out_path = out_dir / f"{EQP_ID}_sensor_{start_s}-{end_s}.parquet"

df.to_parquet(out_path, index=False)
print("저장:", out_path, f"({len(df):,}행)")

## 7. (참고) 여러 날을 하루씩 나눠 받기

기간이 길면 한 번에 받을 때 메모리가 크게 늘어납니다. 하루 단위로 받아 바로 저장하면 메모리를 일정하게 유지할 수 있습니다.
실행하려면 `RUN_DAILY = True`로 바꾸세요.

In [ ]:
RUN_DAILY = False
DAILY_START, DAILY_END = "2026-11-13", "2026-11-15"

if RUN_DAILY:
    for day in pd.date_range(DAILY_START, DAILY_END, freq="D"):
        day_df = load_sensor_data(
            DATA_PATH,
            EQP_ID,
            day,
            day + pd.Timedelta(days=1) - pd.Timedelta(seconds=1),
            columns=COLUMNS,
        )
        day_dir = REPO_ROOT / "output" / day.strftime("%Y-%m-%d")
        day_dir.mkdir(parents=True, exist_ok=True)
        day_df.to_parquet(day_dir / f"{EQP_ID}_sensor_daily.parquet", index=False)
        print(day.date(), f"{len(day_df):,}행 저장")
        del day_df

## 8. (참고) 이미 가진 원본 DataFrame만 변환하기

원본 parquet을 직접 읽었다면 `convert_sensor_df`로 변환만 할 수 있습니다(기간 필터 없음).

In [ ]:
# raw_df = pd.read_parquet("<원본 파일>.parquet")
# wide = convert_sensor_df(raw_df, columns=COLUMNS)